# Example C: Learning a Subgrid Parameterization (The NeuralGCM Analogy)

This notebook demonstrates **end-to-end training of a neural network subgrid parameterization**
through a differentiable shallow water model — the same paradigm used by
[NeuralGCM](https://www.nature.com/articles/s41586-024-07744-y) at full scale.

## The Idea

1. **High-resolution "truth"**: Run the SWE at high resolution (64×64) to generate reference trajectories
2. **Coarse model**: Run at low resolution (16×16) — this has errors because it can't resolve small-scale features
3. **Hybrid model**: Add a small neural network that takes the coarse state as input and outputs a
   correction/forcing term. The entire system (coarse SWE solver + NN) is differentiable
4. **Online training**: Unroll the hybrid model for several timesteps, compare with coarsened
   high-res truth, and backpropagate through the combined physics+NN system

## Why Online Training Matters

- **Offline training** learns the NN correction from instantaneous snapshots, ignoring dynamics.
  When plugged back into the model, offline-trained NNs often cause instability and drift.
- **Online training** backpropagates through the time integration, so the NN learns to
  work *with* the dynamics. This is more stable and accurate over long rollouts.

## Outline

1. Set up high-res and coarse models
2. Generate high-res truth and coarsen it
3. Define the neural network parameterization (Equinox)
4. Train online (end-to-end through the solver)
5. Train offline (on instantaneous snapshots) for comparison
6. Evaluate: coarse-only vs online-trained hybrid vs offline-trained hybrid vs truth

## 1. Imports and Setup

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jrandom
import matplotlib.pyplot as plt
import numpy as np
import equinox as eqx
import optax
from time import perf_counter

jax.config.update("jax_enable_x64", True)

from swm_array_api import initialize_interior, _interior_to_halo, timestep

xp = jnp

print(f"JAX version: {jax.__version__}")
print(f"Equinox version: {eqx.__version__}")
print(f"Default device: {jax.devices()[0]}")
print(f"Float64 enabled: {jax.x64_is_enabled()}")

## 2. Model Configuration

We use two resolutions:
- **High-res (truth)**: 64×64 grid
- **Coarse (model)**: 16×16 grid (4× coarser in each direction)

The coarse model cannot resolve features smaller than 4 grid cells of the truth,
so it accumulates errors over time. The NN will learn to correct these errors.

In [ ]:
# High-resolution (truth) parameters
M_hi, N_hi = 64, 64
dx_hi = 25000.0   # 25 km grid spacing
dy_hi = 25000.0

# Coarse (model) parameters — 4x coarser
COARSEN_FACTOR = M_hi // 16  # = 4
M_lo, N_lo = 16, 16
dx_lo = dx_hi * COARSEN_FACTOR  # 100 km
dy_lo = dy_hi * COARSEN_FACTOR

# Shared physics
dt = 90.0
a = 1000000.0
alpha = 0.001

# Training: unroll window
N_STEPS_TRAIN = 20      # steps per training window (online training)
N_STEPS_EVAL = 100      # steps for evaluation rollout

print(f"High-res grid: {M_hi}×{N_hi}, dx={dx_hi/1000:.0f} km")
print(f"Coarse grid:   {M_lo}×{N_lo}, dx={dx_lo/1000:.0f} km")
print(f"Coarsening factor: {COARSEN_FACTOR}")
print(f"Training window: {N_STEPS_TRAIN} steps ({N_STEPS_TRAIN * dt / 3600:.1f} hours)")
print(f"Evaluation window: {N_STEPS_EVAL} steps ({N_STEPS_EVAL * dt / 3600:.1f} hours)")

## 3. Forward Models and Coarsening Operator

The **coarsening operator** averages blocks of high-res cells to produce the coarse-grid
equivalent. For a 4× coarsening factor, each coarse cell is the mean of a 4×4 block.

In [ ]:
def coarsen(field_hi, factor):
    """Coarsen a high-res interior field by block-averaging.

    Args:
        field_hi: High-res field, shape (M_hi, N_hi).
        factor: Coarsening factor (int).

    Returns:
        Coarse field, shape (M_hi//factor, N_hi//factor).
    """
    M_h, N_h = field_hi.shape
    M_c, N_c = M_h // factor, N_h // factor
    # Reshape into blocks and average
    return field_hi.reshape(M_c, factor, N_c, factor).mean(axis=(1, 3))


def initial_state(u_int, v_int, p_int):
    """Build full state (with halos) from interior fields."""
    u = _interior_to_halo(xp, u_int)
    v = _interior_to_halo(xp, v_int)
    p = _interior_to_halo(xp, p_int)
    return (u, v, p, u, v, p)


def forward_model(u_int, v_int, p_int, n_steps, dx, dy, M, N):
    """Run SWM forward for n_steps. Returns final interior fields."""
    state = initial_state(u_int, v_int, p_int)
    u, v, p, uold, vold, pold = state

    # First step: forward Euler
    unew, vnew, pnew, uold, vold, pold = timestep(
        xp, u, v, p, uold, vold, pold, dx, dy, dt, 0.0, M, N
    )
    carry = (unew, vnew, pnew, uold, vold, pold)

    def scan_step(carry, _):
        u, v, p, uold, vold, pold = carry
        unew, vnew, pnew, uo, vo, po = timestep(
            xp, u, v, p, uold, vold, pold, dx, dy, 2.0 * dt, alpha, M, N
        )
        return (unew, vnew, pnew, uo, vo, po), None

    final, _ = jax.lax.scan(scan_step, carry, None, length=n_steps - 1)
    u_f, v_f, p_f = final[0], final[1], final[2]
    return u_f[1:-1, 1:-1], v_f[1:-1, 1:-1], p_f[1:-1, 1:-1]


def forward_model_trajectory(u_int, v_int, p_int, n_steps, dx, dy, M, N):
    """Run SWM forward, returning interior snapshots at every step.

    Returns:
        (u_traj, v_traj, p_traj): each shape (n_steps, M, N)
    """
    state = initial_state(u_int, v_int, p_int)
    u, v, p, uold, vold, pold = state

    # First step
    unew, vnew, pnew, uold, vold, pold = timestep(
        xp, u, v, p, uold, vold, pold, dx, dy, dt, 0.0, M, N
    )
    carry = (unew, vnew, pnew, uold, vold, pold)

    def scan_step(carry, _):
        u, v, p, uold, vold, pold = carry
        unew, vnew, pnew, uo, vo, po = timestep(
            xp, u, v, p, uold, vold, pold, dx, dy, 2.0 * dt, alpha, M, N
        )
        new_carry = (unew, vnew, pnew, uo, vo, po)
        snap = (unew[1:-1, 1:-1], vnew[1:-1, 1:-1], pnew[1:-1, 1:-1])
        return new_carry, snap

    _, (u_rest, v_rest, p_rest) = jax.lax.scan(
        scan_step, carry, None, length=n_steps - 1
    )

    # Prepend first step
    u_first = unew[1:-1, 1:-1][None]
    v_first = vnew[1:-1, 1:-1][None]
    p_first = pnew[1:-1, 1:-1][None]

    return (
        jnp.concatenate([u_first, u_rest], axis=0),
        jnp.concatenate([v_first, v_rest], axis=0),
        jnp.concatenate([p_first, p_rest], axis=0),
    )


# Test coarsening
u_hi, v_hi, p_hi = initialize_interior(xp, M_hi, N_hi, dx_hi, dy_hi, a)
u_lo_from_hi = coarsen(u_hi, COARSEN_FACTOR)
u_lo_native, v_lo_native, p_lo_native = initialize_interior(xp, M_lo, N_lo, dx_lo, dy_lo, a)

print(f"High-res p shape: {p_hi.shape}")
print(f"Coarsened p shape: {coarsen(p_hi, COARSEN_FACTOR).shape}")
print("Forward model functions defined.")

## 4. Generate High-Res Truth Trajectory

We run the high-res model and save snapshots at every timestep.
These will serve as our training targets (after coarsening).

In [ ]:
print("Running high-res truth simulation...")
t0 = perf_counter()
u_truth_traj, v_truth_traj, p_truth_traj = forward_model_trajectory(
    u_hi, v_hi, p_hi, N_STEPS_EVAL, dx_hi, dy_hi, M_hi, N_hi
)
p_truth_traj.block_until_ready()
t_truth = perf_counter() - t0
print(f"Done in {t_truth:.2f}s. Trajectory shape: {p_truth_traj.shape}")

# Coarsen the truth trajectory to the coarse grid
# vmap over the time dimension
coarsen_vmap = jax.vmap(lambda f: coarsen(f, COARSEN_FACTOR))
u_truth_coarse = coarsen_vmap(u_truth_traj)
v_truth_coarse = coarsen_vmap(v_truth_traj)
p_truth_coarse = coarsen_vmap(p_truth_traj)
print(f"Coarsened truth shape: {p_truth_coarse.shape}")

# Also get coarsened initial condition
u_ic_coarse = coarsen(u_hi, COARSEN_FACTOR)
v_ic_coarse = coarsen(v_hi, COARSEN_FACTOR)
p_ic_coarse = coarsen(p_hi, COARSEN_FACTOR)
print(f"Coarsened IC p range: [{float(p_ic_coarse.min()):.1f}, {float(p_ic_coarse.max()):.1f}]")

## 5. Baseline: Coarse Model Without Correction

Run the coarse model from the coarsened initial condition and see how it diverges
from the coarsened truth.

In [ ]:
print("Running coarse-only simulation...")
u_coarse_traj, v_coarse_traj, p_coarse_traj = forward_model_trajectory(
    u_ic_coarse, v_ic_coarse, p_ic_coarse, N_STEPS_EVAL, dx_lo, dy_lo, M_lo, N_lo
)
p_coarse_traj.block_until_ready()

# Compute RMS error at each timestep
def rms_error_trajectory(pred_traj, truth_traj):
    """RMS error at each timestep."""
    diff = pred_traj - truth_traj
    return jnp.sqrt(jnp.mean(diff**2, axis=(1, 2)))

rms_coarse_p = rms_error_trajectory(p_coarse_traj, p_truth_coarse)
print(f"Coarse model p RMS error: initial={float(rms_coarse_p[0]):.2f}, "
      f"final={float(rms_coarse_p[-1]):.2f}")

## 6. Neural Network Parameterization

We define a small convolutional neural network using [Equinox](https://docs.kidger.site/equinox/),
a JAX library for neural networks built on pure functions.

The NN takes the coarse state (u, v, p) as a 3-channel input and outputs
a 3-channel correction (du, dv, dp) that will be added as a forcing term
at each timestep.

**Key design choices:**
- Circular (periodic) padding to match the periodic boundary conditions
- Small architecture (3 layers, 16 channels) — this is a demo, not a production model
- Output initialized near zero so the hybrid model starts close to the coarse model

In [ ]:
class SubgridNN(eqx.Module):
    """Small CNN for subgrid correction.

    Takes (u, v, p) interior fields, returns (du, dv, dp) corrections.
    Uses circular padding for periodic BCs.
    """
    conv1: eqx.nn.Conv2d
    conv2: eqx.nn.Conv2d
    conv3: eqx.nn.Conv2d
    # Scaling factor: learnable output magnitude
    output_scale: jax.Array

    def __init__(self, key, n_channels=16):
        k1, k2, k3 = jrandom.split(key, 3)
        # 3 input channels (u, v, p), hidden channels, 3 output channels (du, dv, dp)
        self.conv1 = eqx.nn.Conv2d(3, n_channels, kernel_size=3, padding=0, key=k1)
        self.conv2 = eqx.nn.Conv2d(n_channels, n_channels, kernel_size=3, padding=0, key=k2)
        self.conv3 = eqx.nn.Conv2d(n_channels, 3, kernel_size=3, padding=0, key=k3)
        # Start with small output scale so initial corrections are near zero
        self.output_scale = jnp.array(0.01)

    def __call__(self, u_int, v_int, p_int):
        """Compute subgrid correction.

        Args:
            u_int, v_int, p_int: Interior fields, each shape (M, N).

        Returns:
            du, dv, dp: Correction fields, each shape (M, N).
        """
        # Stack into (3, M, N) for conv2d
        x = jnp.stack([u_int, v_int, p_int], axis=0)

        # Layer 1: circular pad + conv + tanh
        x = jnp.pad(x, ((0, 0), (1, 1), (1, 1)), mode='wrap')
        x = self.conv1(x)
        x = jnp.tanh(x)

        # Layer 2: circular pad + conv + tanh
        x = jnp.pad(x, ((0, 0), (1, 1), (1, 1)), mode='wrap')
        x = self.conv2(x)
        x = jnp.tanh(x)

        # Layer 3: circular pad + conv (linear output)
        x = jnp.pad(x, ((0, 0), (1, 1), (1, 1)), mode='wrap')
        x = self.conv3(x)

        # Scale output
        x = x * self.output_scale

        return x[0], x[1], x[2]  # du, dv, dp


# Test the network
key = jrandom.PRNGKey(0)
nn = SubgridNN(key)
du, dv, dp = nn(u_ic_coarse, v_ic_coarse, p_ic_coarse)

n_params = sum(x.size for x in jax.tree.leaves(eqx.filter(nn, eqx.is_array)))
print(f"NN parameter count: {n_params}")
print(f"Initial correction magnitudes: du={float(jnp.abs(du).max()):.2e}, "
      f"dv={float(jnp.abs(dv).max()):.2e}, dp={float(jnp.abs(dp).max()):.2e}")

## 7. Hybrid Model: Coarse SWE + NN Correction

The hybrid model runs a standard SWE timestep and then adds the NN correction
to the new state. The correction acts as a learned forcing/tendency term.

Crucially, this entire model — physics + NN — is differentiable, so we can
backpropagate through the time integration to train the NN weights.

In [ ]:
def hybrid_timestep(nn_model, u, v, p, uold, vold, pold, dt_val, alpha_val):
    """One timestep of the hybrid model: SWE physics + NN correction.

    The NN correction is applied to the interior of the new state after
    the physics timestep, scaled by dt for dimensional consistency.
    """
    # Physics step
    unew, vnew, pnew, uold_new, vold_new, pold_new = timestep(
        xp, u, v, p, uold, vold, pold, dx_lo, dy_lo, dt_val, alpha_val, M_lo, N_lo
    )

    # NN correction on interior
    du, dv, dp = nn_model(unew[1:-1, 1:-1], vnew[1:-1, 1:-1], pnew[1:-1, 1:-1])

    # Apply correction (scaled by dt for physical consistency)
    unew_int = unew[1:-1, 1:-1] + du * dt_val
    vnew_int = vnew[1:-1, 1:-1] + dv * dt_val
    pnew_int = pnew[1:-1, 1:-1] + dp * dt_val

    # Rebuild with halos
    unew = _interior_to_halo(xp, unew_int)
    vnew = _interior_to_halo(xp, vnew_int)
    pnew = _interior_to_halo(xp, pnew_int)

    return unew, vnew, pnew, uold_new, vold_new, pold_new


def hybrid_forward_trajectory(nn_model, u_int, v_int, p_int, n_steps):
    """Run hybrid model forward, returning interior snapshots at every step."""
    u = _interior_to_halo(xp, u_int)
    v = _interior_to_halo(xp, v_int)
    p = _interior_to_halo(xp, p_int)
    uold, vold, pold = u, v, p

    # First step: forward Euler (no time filter)
    unew, vnew, pnew, uold, vold, pold = hybrid_timestep(
        nn_model, u, v, p, uold, vold, pold, dt, 0.0
    )
    carry = (unew, vnew, pnew, uold, vold, pold)

    def scan_step(carry, _):
        u, v, p, uold, vold, pold = carry
        unew, vnew, pnew, uo, vo, po = hybrid_timestep(
            nn_model, u, v, p, uold, vold, pold, 2.0 * dt, alpha
        )
        snap = (unew[1:-1, 1:-1], vnew[1:-1, 1:-1], pnew[1:-1, 1:-1])
        return (unew, vnew, pnew, uo, vo, po), snap

    _, (u_rest, v_rest, p_rest) = jax.lax.scan(
        scan_step, carry, None, length=n_steps - 1
    )

    u_first = unew[1:-1, 1:-1][None]
    v_first = vnew[1:-1, 1:-1][None]
    p_first = pnew[1:-1, 1:-1][None]

    return (
        jnp.concatenate([u_first, u_rest], axis=0),
        jnp.concatenate([v_first, v_rest], axis=0),
        jnp.concatenate([p_first, p_rest], axis=0),
    )


print("Hybrid model functions defined.")

## 8. Online Training: End-to-End Through the Solver

This is the key demonstration. We:
1. Unroll the hybrid model for `N_STEPS_TRAIN` timesteps
2. Compare the trajectory with coarsened truth
3. Backpropagate through the entire time integration to compute ∇loss w.r.t. NN weights
4. Update NN weights with Adam optimizer

The gradient flows backward through: loss → final state → NN correction → physics step → ... → initial state.
This is exactly what NeuralGCM does at scale.

In [ ]:
def online_loss(nn_model, u_int, v_int, p_int, u_target, v_target, p_target):
    """Online training loss: unroll hybrid model and compare trajectory to truth.

    Args:
        nn_model: The SubgridNN model.
        u_int, v_int, p_int: Initial condition (coarse, interior).
        u_target, v_target, p_target: Coarsened truth trajectory, shape (N_STEPS_TRAIN, M_lo, N_lo).

    Returns:
        Scalar MSE loss over the trajectory.
    """
    u_pred, v_pred, p_pred = hybrid_forward_trajectory(
        nn_model, u_int, v_int, p_int, N_STEPS_TRAIN
    )

    # MSE over trajectory
    loss = (
        jnp.mean((u_pred - u_target) ** 2)
        + jnp.mean((v_pred - v_target) ** 2)
        + jnp.mean((p_pred - p_target) ** 2)
    )
    return loss


# Training targets: coarsened truth trajectory for the training window
u_train_target = p_truth_coarse[:N_STEPS_TRAIN]  # Oops — let's fix: use u, v, p
u_train_target = u_truth_coarse[:N_STEPS_TRAIN]
v_train_target = v_truth_coarse[:N_STEPS_TRAIN]
p_train_target = p_truth_coarse[:N_STEPS_TRAIN]

print(f"Training target shapes: u={u_train_target.shape}, v={v_train_target.shape}, p={p_train_target.shape}")

In [ ]:
# Set up optimizer
learning_rate = 1e-3
optimizer = optax.adam(learning_rate)

# Initialize model and optimizer state
key = jrandom.PRNGKey(42)
nn_online = SubgridNN(key)
opt_state = optimizer.init(eqx.filter(nn_online, eqx.is_array))

# JIT-compiled training step
@eqx.filter_jit
def train_step_online(nn_model, opt_state):
    """One online training step: compute loss + gradient, update weights."""
    loss, grads = eqx.filter_value_and_grad(online_loss)(
        nn_model, u_ic_coarse, v_ic_coarse, p_ic_coarse,
        u_train_target, v_train_target, p_train_target
    )
    updates, new_opt_state = optimizer.update(
        grads, opt_state, eqx.filter(nn_model, eqx.is_array)
    )
    new_model = eqx.apply_updates(nn_model, updates)
    return new_model, new_opt_state, loss


print("Training step compiled. Starting online training...")

In [ ]:
# Online training loop
N_EPOCHS = 100
online_loss_history = []

print(f"Training for {N_EPOCHS} epochs (backpropagating through {N_STEPS_TRAIN} timesteps)...")
print(f"{'Epoch':>6s}  {'Loss':>12s}  {'Time':>8s}")
print("-" * 32)

t0_total = perf_counter()
for epoch in range(N_EPOCHS):
    t0 = perf_counter()
    nn_online, opt_state, loss = train_step_online(nn_online, opt_state)
    loss_val = float(loss)
    t_epoch = perf_counter() - t0
    online_loss_history.append(loss_val)

    if epoch % 10 == 0 or epoch == N_EPOCHS - 1:
        print(f"{epoch:6d}  {loss_val:12.6e}  {t_epoch:7.3f}s")

t_total = perf_counter() - t0_total
print(f"\nOnline training complete in {t_total:.1f}s")
print(f"Final loss: {online_loss_history[-1]:.6e}")

## 9. Offline Training: Snapshot-Based (For Comparison)

For contrast, we also train a NN using **offline** (snapshot-based) training.
Here, the NN learns to predict the instantaneous correction needed at each
timestep, without unrolling through the dynamics.

The offline target is the "tendency error": the difference between what the
coarse model produces and what the coarsened truth shows at each timestep.

In [ ]:
# Compute offline training targets: tendency error at each step
# For each timestep, the "ideal correction" is:
#   target_correction = (coarsened_truth[t+1] - coarse_model_prediction[t+1]) / dt
# We approximate this using the coarse model trajectory.

# The offline target is the difference between coarsened truth and coarse model
# at each timestep, normalized by dt (so it's a tendency)
offline_du_target = (u_truth_coarse[:N_STEPS_TRAIN] - u_coarse_traj[:N_STEPS_TRAIN]) / (2.0 * dt)
offline_dv_target = (v_truth_coarse[:N_STEPS_TRAIN] - v_coarse_traj[:N_STEPS_TRAIN]) / (2.0 * dt)
offline_dp_target = (p_truth_coarse[:N_STEPS_TRAIN] - p_coarse_traj[:N_STEPS_TRAIN]) / (2.0 * dt)

# Use the coarse model states as inputs for offline training
offline_u_input = u_coarse_traj[:N_STEPS_TRAIN]
offline_v_input = v_coarse_traj[:N_STEPS_TRAIN]
offline_p_input = p_coarse_traj[:N_STEPS_TRAIN]

print(f"Offline training data: {N_STEPS_TRAIN} snapshots")
print(f"Tendency error magnitudes: "
      f"du={float(jnp.std(offline_du_target)):.2e}, "
      f"dv={float(jnp.std(offline_dv_target)):.2e}, "
      f"dp={float(jnp.std(offline_dp_target)):.2e}")

In [ ]:
def offline_loss(nn_model, u_inputs, v_inputs, p_inputs,
                 du_targets, dv_targets, dp_targets):
    """Offline training loss: predict instantaneous correction at each snapshot."""
    def single_loss(u_in, v_in, p_in, du_tgt, dv_tgt, dp_tgt):
        du_pred, dv_pred, dp_pred = nn_model(u_in, v_in, p_in)
        return (
            jnp.mean((du_pred - du_tgt) ** 2)
            + jnp.mean((dv_pred - dv_tgt) ** 2)
            + jnp.mean((dp_pred - dp_tgt) ** 2)
        )

    # vmap over snapshots
    losses = jax.vmap(single_loss)(u_inputs, v_inputs, p_inputs,
                                    du_targets, dv_targets, dp_targets)
    return jnp.mean(losses)


# Initialize a separate model for offline training
key = jrandom.PRNGKey(42)  # same init for fair comparison
nn_offline = SubgridNN(key)
opt_state_offline = optimizer.init(eqx.filter(nn_offline, eqx.is_array))


@eqx.filter_jit
def train_step_offline(nn_model, opt_state):
    loss, grads = eqx.filter_value_and_grad(offline_loss)(
        nn_model, offline_u_input, offline_v_input, offline_p_input,
        offline_du_target, offline_dv_target, offline_dp_target
    )
    updates, new_opt_state = optimizer.update(
        grads, opt_state, eqx.filter(nn_model, eqx.is_array)
    )
    new_model = eqx.apply_updates(nn_model, updates)
    return new_model, new_opt_state, loss


# Offline training loop
offline_loss_history = []

print(f"Offline training for {N_EPOCHS} epochs...")
print(f"{'Epoch':>6s}  {'Loss':>12s}  {'Time':>8s}")
print("-" * 32)

t0_total = perf_counter()
for epoch in range(N_EPOCHS):
    t0 = perf_counter()
    nn_offline, opt_state_offline, loss = train_step_offline(nn_offline, opt_state_offline)
    loss_val = float(loss)
    t_epoch = perf_counter() - t0
    offline_loss_history.append(loss_val)

    if epoch % 10 == 0 or epoch == N_EPOCHS - 1:
        print(f"{epoch:6d}  {loss_val:12.6e}  {t_epoch:7.3f}s")

t_total = perf_counter() - t0_total
print(f"\nOffline training complete in {t_total:.1f}s")

## 10. Evaluation: Coarse vs Online Hybrid vs Offline Hybrid

Now we compare all three models over the full evaluation window (longer than training).
This tests generalization — can the NN corrections remain stable beyond the training window?

In [ ]:
print("Running evaluation rollouts...")

# Online-trained hybrid
u_online_traj, v_online_traj, p_online_traj = hybrid_forward_trajectory(
    nn_online, u_ic_coarse, v_ic_coarse, p_ic_coarse, N_STEPS_EVAL
)
p_online_traj.block_until_ready()

# Offline-trained hybrid
u_offline_traj, v_offline_traj, p_offline_traj = hybrid_forward_trajectory(
    nn_offline, u_ic_coarse, v_ic_coarse, p_ic_coarse, N_STEPS_EVAL
)
p_offline_traj.block_until_ready()

# Compute RMS errors
rms_coarse_p = rms_error_trajectory(p_coarse_traj, p_truth_coarse)
rms_online_p = rms_error_trajectory(p_online_traj, p_truth_coarse)
rms_offline_p = rms_error_trajectory(p_offline_traj, p_truth_coarse)

rms_coarse_u = rms_error_trajectory(u_coarse_traj, u_truth_coarse)
rms_online_u = rms_error_trajectory(u_online_traj, u_truth_coarse)
rms_offline_u = rms_error_trajectory(u_offline_traj, u_truth_coarse)

print(f"\nFinal RMS errors (pressure, step {N_STEPS_EVAL}):")
print(f"  Coarse only:     {float(rms_coarse_p[-1]):.4f}")
print(f"  Offline hybrid:  {float(rms_offline_p[-1]):.4f}")
print(f"  Online hybrid:   {float(rms_online_p[-1]):.4f}")
print(f"\nFinal RMS errors (u-velocity, step {N_STEPS_EVAL}):")
print(f"  Coarse only:     {float(rms_coarse_u[-1]):.4f}")
print(f"  Offline hybrid:  {float(rms_offline_u[-1]):.4f}")
print(f"  Online hybrid:   {float(rms_online_u[-1]):.4f}")

## 11. Visualize Results

In [ ]:
# --- Training convergence ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(online_loss_history, 'b-', linewidth=2, label='Online (end-to-end)')
ax.semilogy(offline_loss_history, 'r--', linewidth=2, label='Offline (snapshot)')
ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Training Loss', fontsize=13)
ax.set_title('Training Convergence: Online vs Offline', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- RMS error over time ---
timesteps = np.arange(1, N_STEPS_EVAL + 1)
hours = timesteps * dt / 3600

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pressure
ax1.plot(hours, np.array(rms_coarse_p), 'k-', linewidth=2, label='Coarse only')
ax1.plot(hours, np.array(rms_offline_p), 'r--', linewidth=2, label='Offline hybrid')
ax1.plot(hours, np.array(rms_online_p), 'b-', linewidth=2, label='Online hybrid')
ax1.axvline(N_STEPS_TRAIN * dt / 3600, color='gray', linestyle=':', alpha=0.7, label='Training window')
ax1.set_xlabel('Time (hours)', fontsize=13)
ax1.set_ylabel('RMS Error (pressure)', fontsize=13)
ax1.set_title('Pressure Field Error vs Time', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Velocity
ax2.plot(hours, np.array(rms_coarse_u), 'k-', linewidth=2, label='Coarse only')
ax2.plot(hours, np.array(rms_offline_u), 'r--', linewidth=2, label='Offline hybrid')
ax2.plot(hours, np.array(rms_online_u), 'b-', linewidth=2, label='Online hybrid')
ax2.axvline(N_STEPS_TRAIN * dt / 3600, color='gray', linestyle=':', alpha=0.7, label='Training window')
ax2.set_xlabel('Time (hours)', fontsize=13)
ax2.set_ylabel('RMS Error (u-velocity)', fontsize=13)
ax2.set_title('U-Velocity Field Error vs Time', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Snapshot comparison at evaluation endpoint ---
t_idx = -1  # last timestep

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

# Row 1: Pressure fields
fields_p = [
    (p_truth_coarse[t_idx], 'Coarsened truth'),
    (p_coarse_traj[t_idx], 'Coarse only'),
    (p_offline_traj[t_idx], 'Offline hybrid'),
    (p_online_traj[t_idx], 'Online hybrid'),
]
vmin_p = float(p_truth_coarse[t_idx].min())
vmax_p = float(p_truth_coarse[t_idx].max())

for j, (field, title) in enumerate(fields_p):
    im = axes[0, j].imshow(np.array(field), cmap='viridis', origin='lower',
                           vmin=vmin_p, vmax=vmax_p)
    axes[0, j].set_title(title, fontsize=12)
    plt.colorbar(im, ax=axes[0, j], shrink=0.8)

# Row 2: Error fields (difference from truth)
errors_p = [
    (p_truth_coarse[t_idx] - p_truth_coarse[t_idx], 'Truth (reference)'),
    (p_coarse_traj[t_idx] - p_truth_coarse[t_idx], 'Coarse error'),
    (p_offline_traj[t_idx] - p_truth_coarse[t_idx], 'Offline error'),
    (p_online_traj[t_idx] - p_truth_coarse[t_idx], 'Online error'),
]
vmax_err = max(float(jnp.abs(p_coarse_traj[t_idx] - p_truth_coarse[t_idx]).max()),
               float(jnp.abs(p_offline_traj[t_idx] - p_truth_coarse[t_idx]).max()),
               float(jnp.abs(p_online_traj[t_idx] - p_truth_coarse[t_idx]).max()))

for j, (field, title) in enumerate(errors_p):
    im = axes[1, j].imshow(np.array(field), cmap='RdBu_r', origin='lower',
                           vmin=-vmax_err, vmax=vmax_err)
    rms = float(jnp.sqrt(jnp.mean(field**2)))
    axes[1, j].set_title(f'{title}\n(RMS={rms:.2f})', fontsize=12)
    plt.colorbar(im, ax=axes[1, j], shrink=0.8)

fig.suptitle(f'Pressure at step {N_STEPS_EVAL} ({N_STEPS_EVAL * dt / 3600:.1f} hours)',
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- NN correction field visualization ---
# What has the online-trained NN learned?
du_online, dv_online, dp_online = nn_online(u_ic_coarse, v_ic_coarse, p_ic_coarse)
du_offline, dv_offline, dp_offline = nn_offline(u_ic_coarse, v_ic_coarse, p_ic_coarse)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for col, (du, dv, dp, label) in enumerate([
    (du_online, dv_online, dp_online, 'Online-trained'),
    (du_offline, dv_offline, dp_offline, 'Offline-trained'),
]):
    for row, (field, name) in enumerate([(du, 'du'), (dv, 'dv'), (dp, 'dp')]):
        ax_idx = col * 3 + row
        ax = axes[col, row]
        fdata = np.array(field)
        vmax = np.abs(fdata).max()
        if vmax == 0:
            vmax = 1.0
        im = ax.imshow(fdata, cmap='RdBu_r', origin='lower', vmin=-vmax, vmax=vmax)
        ax.set_title(f'{label}: {name}\n(max={vmax:.2e})', fontsize=12)
        plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('Learned NN Corrections at Initial Condition', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 12. Quantitative Summary

In [ ]:
# Summary table
print("=" * 72)
print(f"{'':30s} {'Pressure':>12s} {'U-velocity':>12s} {'V-velocity':>12s}")
print("=" * 72)

for name, u_traj, v_traj, p_traj in [
    ('Coarse only', u_coarse_traj, v_coarse_traj, p_coarse_traj),
    ('Offline hybrid', u_offline_traj, v_offline_traj, p_offline_traj),
    ('Online hybrid', u_online_traj, v_online_traj, p_online_traj),
]:
    rms_p = float(rms_error_trajectory(p_traj, p_truth_coarse)[-1])
    rms_u = float(rms_error_trajectory(u_traj, u_truth_coarse)[-1])
    rms_v = float(rms_error_trajectory(v_traj, v_truth_coarse)[-1])
    print(f"{name:30s} {rms_p:12.4f} {rms_u:12.6f} {rms_v:12.6f}")

print("=" * 72)

# Improvement ratios
coarse_p_final = float(rms_error_trajectory(p_coarse_traj, p_truth_coarse)[-1])
online_p_final = float(rms_error_trajectory(p_online_traj, p_truth_coarse)[-1])
offline_p_final = float(rms_error_trajectory(p_offline_traj, p_truth_coarse)[-1])

print(f"\nOnline hybrid improvement over coarse: {coarse_p_final / online_p_final:.1f}x (pressure)")
print(f"Offline hybrid improvement over coarse: {coarse_p_final / offline_p_final:.1f}x (pressure)")
if online_p_final < offline_p_final:
    print(f"Online training advantage over offline: {offline_p_final / online_p_final:.1f}x better")
else:
    print(f"Offline training advantage over online: {online_p_final / offline_p_final:.1f}x better")

## Summary

**What we demonstrated:**

1. A **hybrid physics-ML model**: a coarse SWE solver augmented with a small CNN that
   learns subgrid corrections. The entire system — physics + neural network — is
   differentiable thanks to JAX.

2. **Online (end-to-end) training** backpropagates through the time integration,
   so the NN learns corrections that work well *within the dynamics*. This is the
   same approach used by [NeuralGCM](https://www.nature.com/articles/s41586-024-07744-y).

3. **Offline (snapshot) training** learns corrections independently from dynamics.
   While faster to train, offline corrections can cause drift when coupled to the solver.

4. The hybrid model with online training **outperforms** the coarse-only model,
   demonstrating that learned subgrid parameterizations can meaningfully improve
   coarse simulations.

**Connection to NeuralGCM:**

This is NeuralGCM in miniature. NeuralGCM couples a differentiable dynamical core
("Dinosaur", also in JAX) with neural network parameterizations for clouds, radiation,
and subgrid processes. The key insight is the same: by making the physics differentiable,
you can train the ML components end-to-end, producing a hybrid model that is more
stable and accurate than either physics or ML alone.

**Key enabler:** The `swm_array_api.py` implementation uses a functional, mutation-free
style that is directly compatible with JAX's AD. Combined with Equinox for the neural
network, the entire hybrid model is differentiable without any custom adjoint code.